In [1]:
import json
import sys
import time
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.sentiment_recovery import (
    DEFAULT_MULTILINGUAL_MODEL,
    build_char_tfidf_model,
    build_word_tfidf_model,
    cross_validate_group_model,
    dataset_summary,
    fit_and_predict_model,
    run_majority_baseline,
    run_multilingual_pipeline_baseline,
)


In [2]:
prepared_dir = ROOT / "data" / "nlp" / "prepared"
report_path = prepared_dir / "prep_report.json"
all_path = prepared_dir / "model2_reviews_labeled.csv"
train_path = prepared_dir / "model2_train.csv"
val_path = prepared_dir / "model2_val.csv"
test_path = prepared_dir / "model2_test.csv"

all_df = pd.read_csv(all_path)
train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
test_df = pd.read_csv(test_path)

prep_report = json.loads(report_path.read_text(encoding="utf-8")) if report_path.exists() else {}

print("all:", all_df.shape, "train:", train_df.shape, "val:", val_df.shape, "test:", test_df.shape)
display(pd.DataFrame(dataset_summary(train_df, val_df, test_df)).T)
display(pd.DataFrame([prep_report.get("counts", {}).get("model2", {})]).T)


all: (1560, 10) train: (1217, 10) val: (159, 10) test: (184, 10)


,rows,label_distribution,language_distribution,unique_groups
train,1217,"{'positive': 462, 'negative': 411, 'neutral': ...","{'fr': 645, 'en': 572}",303
val,159,"{'positive': 86, 'neutral': 59, 'negative': 14}","{'en': 137, 'fr': 22}",42
test,184,"{'neutral': 103, 'positive': 68, 'negative': 13}","{'fr': 125, 'en': 59}",39


,0
all_rows,1560
unique_clean_text,384
unique_groups,384
groups_per_class,"{'positive': 128, 'negative': 128, 'neutral': ..."
train_rows,1217
val_rows,159
test_rows,184
source_distribution,"{'reviews_metadata_csv': 1200, 'synthetic_temp..."
label_source_distribution,"{'metadata_label': 1200, 'synthetic_template_l..."
sentiment_distribution,"{'positive': 616, 'neutral': 506, 'negative': ..."


In [3]:
artifact_dir = ROOT / "artifacts" / "reports" / "nlp_sentiment"
artifact_dir.mkdir(parents=True, exist_ok=True)

def save_eval_artifacts(name: str, metrics: dict, preds_df: pd.DataFrame) -> None:
    preds_df.to_csv(artifact_dir / f"{name}_predictions.csv", index=False)
    (artifact_dir / f"{name}_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")


## TF-IDF Baselines

In [4]:
rows = []

majority_val_metrics, majority_val_preds = run_majority_baseline(train_df, val_df, "majority_val")
majority_test_metrics, majority_test_preds = run_majority_baseline(train_df, test_df, "majority_test")
save_eval_artifacts("majority_val", majority_val_metrics, majority_val_preds)
save_eval_artifacts("majority_test", majority_test_metrics, majority_test_preds)
rows.extend([majority_val_metrics, majority_test_metrics])

for model_name, builder in [
    ("tfidf_word", build_word_tfidf_model),
    ("tfidf_char", build_char_tfidf_model),
]:
    t0 = time.time()
    val_metrics, val_preds = fit_and_predict_model(builder(), train_df, val_df, f"{model_name}_val")
    test_metrics, test_preds = fit_and_predict_model(builder(), train_df, test_df, f"{model_name}_test")
    elapsed = round(time.time() - t0, 2)
    val_metrics["train_time_sec"] = elapsed
    test_metrics["train_time_sec"] = elapsed
    save_eval_artifacts(f"{model_name}_val", val_metrics, val_preds)
    save_eval_artifacts(f"{model_name}_test", test_metrics, test_preds)
    rows.extend([val_metrics, test_metrics])

baseline_results = pd.DataFrame(rows)[["split", "accuracy", "f1_macro", "f1_weighted"]]
display(baseline_results.sort_values("split").reset_index(drop=True))


,split,accuracy,f1_macro,f1_weighted
0,majority_test,0.369565,0.179894,0.199448
1,majority_val,0.540881,0.234014,0.379720
2,tfidf_char_test,0.695652,0.695420,0.621656
3,tfidf_char_val,1.000000,1.000000,1.000000
4,tfidf_word_test,0.195652,0.232127,0.167684
5,tfidf_word_val,1.000000,1.000000,1.000000


## Grouped Cross-Validation

In [5]:
cv_rows = []
for model_name, builder in [
    ("tfidf_word", build_word_tfidf_model),
    ("tfidf_char", build_char_tfidf_model),
]:
    cv_report = cross_validate_group_model(all_df, model_factory=builder, max_splits=5)
    cv_report["model"] = model_name
    cv_rows.append(cv_report)
    (artifact_dir / f"{model_name}_group_cv.json").write_text(json.dumps(cv_report, indent=2), encoding="utf-8")

display(pd.DataFrame(cv_rows)[["model", "status", "fold_count", "f1_macro_mean", "f1_macro_std", "fold_scores"]])


,model,status,fold_count,f1_macro_mean,f1_macro_std,fold_scores
0,tfidf_word,ok,5,0.471848,0.119696,"[0.6758670179722811, 0.5391449819352481, 0.392..."
1,tfidf_char,ok,5,0.836620,0.177772,"[1.0, 1.0, 0.8710754266309823, 0.7942198646423..."


## Pretrained Multilingual Baseline

In [ ]:
pipeline_rows = []
for split_name, df in [("pipeline_val", val_df), ("pipeline_test", test_df)]:
    try:
        metrics, preds = run_multilingual_pipeline_baseline(
            df,
            model_name=DEFAULT_MULTILINGUAL_MODEL,
            split_name=split_name,
        )
        save_eval_artifacts(split_name, metrics, preds)
        pipeline_rows.append(metrics)
    except Exception as exc:
        pipeline_rows.append({
            "split": split_name,
            "status": f"skipped: {exc}",
        })

display(pd.DataFrame(pipeline_rows))

In [7]:
import joblib

# Choose the best local classical model from grouped CV
cv_df = pd.DataFrame(cv_rows)
eligible_cv = cv_df.loc[cv_df["status"] == "ok"].copy()

if eligible_cv.empty:
    raise RuntimeError("No trainable local model is available to save.")

best_row = eligible_cv.sort_values(
    ["f1_macro_mean", "f1_macro_std"],
    ascending=[False, True],
).iloc[0]

best_model_name = best_row["model"]

model_builders = {
    "tfidf_word": build_word_tfidf_model,
    "tfidf_char": build_char_tfidf_model,
}

if best_model_name not in model_builders:
    raise RuntimeError(f"Unsupported best model: {best_model_name}")

# Refit on train + val after model selection
final_train_df = pd.concat([train_df, val_df], ignore_index=True)
X_final = final_train_df["clean_text"].fillna("").astype(str)
y_final = final_train_df["sentiment_label"].fillna("neutral").astype(str)
mask = X_final.str.len().gt(0)

best_model = model_builders[best_model_name]()
best_model.fit(X_final[mask], y_final[mask])

models_dir = ROOT / "artifacts" / "models"
reports_dir = ROOT / "artifacts" / "reports" / "nlp_sentiment"
models_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)

model_path = models_dir / f"{best_model_name}_sentiment.joblib"
report_path = reports_dir / "best_sentiment_model_report.json"

joblib.dump(best_model, model_path)

best_model_report = {
    "selected_model": best_model_name,
    "selection_source": "grouped_cv",
    "model_path": str(model_path),
    "train_rows": int(len(train_df)),
    "val_rows": int(len(val_df)),
    "final_fit_rows": int(mask.sum()),
    "cv_f1_macro_mean": float(best_row["f1_macro_mean"]),
    "cv_f1_macro_std": float(best_row["f1_macro_std"]),
    "label_order": ["negative", "neutral", "positive"],
}

report_path.write_text(json.dumps(best_model_report, indent=2), encoding="utf-8")

print(f"Saved best model: {best_model_name}")
print(f"Model path: {model_path}")
print(f"Report path: {report_path}")
best_model_report


Saved best model: tfidf_char
Model path: C:\Users\ASUS\Desktop\Projects\EstateMind\artifacts\models\tfidf_char_sentiment.joblib
Report path: C:\Users\ASUS\Desktop\Projects\EstateMind\artifacts\reports\nlp_sentiment\best_sentiment_model_report.json


{'selected_model': 'tfidf_char',
 'selection_source': 'grouped_cv',
 'model_path': 'C:\\Users\\ASUS\\Desktop\\Projects\\EstateMind\\artifacts\\models\\tfidf_char_sentiment.joblib',
 'train_rows': 1217,
 'val_rows': 159,
 'final_fit_rows': 1376,
 'cv_f1_macro_mean': 0.8366200571223086,
 'cv_f1_macro_std': 0.17777176633976954,
 'label_order': ['negative', 'neutral', 'positive']}